# Raspador do Portal de Periódicos da CAPES

Este notebook demonstra o uso do raspador para o buscador público do Portal de Periódicos da CAPES, que indexa metadados bibliográficos via integração com o OpenAlex.

**Fonte:** [periodicos.capes.gov.br](https://www.periodicos.capes.gov.br/index.php/acervo/buscador.html)

**Tipo de dados:** Metadados de artigos, livros e capítulos (autores, ano, revista, instituição, tópicos, resumo, DOI). O acesso aos textos completos exige login institucional via CAFe; o raspador coleta apenas os metadados da página pública de busca.

## Importação

In [1]:
import raspe

## Uso Básico

O raspador da CAPES usa o parâmetro `pesquisa` para indicar o termo de busca. O parâmetro `paginas` aceita um `range` para limitar quantas páginas (de 30 resultados cada) serão coletadas. Termos genéricos como `"saúde"` retornam milhões de registros, então em demonstrações é recomendável restringir o intervalo.

In [2]:
# Busca simples por um termo
scraper = raspe.capes()
dados = scraper.raspar(pesquisa="natjus", paginas=range(1, 2))

print(f"Total de registros: {len(dados)}")
dados[["titulo", "autores", "ano", "revista"]].head()

2026-05-18 19:06:29,465 - capes - INFO - Iniciando raspagem com parâmetros {'pesquisa': 'natjus', 'paginas': range(1, 2)}


2026-05-18 19:06:29,466 - capes - DEBUG - Definindo consulta


2026-05-18 19:06:29,466 - capes - DEBUG - {'q': 'all:contains(natjus)', 'mode': 'advanced', 'source': 'all'}


2026-05-18 19:06:29,466 - capes - DEBUG - Definindo n_pags


2026-05-18 19:06:29,466 - capes - DEBUG - Enviando requisição inicial com retry automático


...

Processando documentos: 100%|██████████| 1/1 [00:00<00:00, 16.04it/s]


2026-05-18 19:07:08,049 - capes - DEBUG - Adicionada coluna termo_busca=natjus aos resultados


2026-05-18 19:07:08,049 - capes - INFO - Raspagem finalizada, limpando diretório /tmp/tmpq7ko4nbl/capes/20260518190704


Total de registros: 20


,titulo,autores,ano,revista
0,Judicialização de produtos à base de canabidio...,Ronaldo Portela; Daniel Marques Mota; Paulo Jo...,2023,Cadernos de Saúde Pública
1,O direito à saúde e a dispensação judicial de ...,João Paulo Kulczynski Forster; Najwa Dagash; P...,2020,
2,Notas técnicas para judicialização de anticoag...,Nívea Aparecida de Almeida; Mara Luiza de Paiv...,2022,Medicina (Ribeirão Preto)
3,Análise das notas técnicas do Núcleo de Apoio ...,Pedro Paulo Ribeiro Silva Cunha; Leonardo Bonf...,2023,
4,"DIREITOS DA PERSONALIDADE, BANCOS DE DADOS E I...",Ricardo da Silveira e Silva; Rodrigo Valente G...,2024,Revista de Processo Jurisdição e Efetividade d...


## Parâmetros Disponíveis

| Parâmetro | Tipo | Descrição |
|-----------|------|------------|
| `pesquisa` | `str` ou `list[str]` | Termo(s) de busca. Internamente, vira `q=all:contains({termo})` na URL. Obrigatório. |
| `paginas` | `range` | Intervalo de páginas a raspar (cada página tem 30 resultados). Opcional — sem isto, percorre todas as páginas disponíveis. |

## Colunas Retornadas

| Coluna | Descrição |
|--------|------------|
| `id` | Identificador `Work` do OpenAlex (ex.: `W4387465478`) |
| `tipo` | Tipo de obra (artigo, livro, capítulo, etc.) |
| `titulo` | Título do registro |
| `link` | URL para a página de detalhe no portal da CAPES |
| `autores` | Lista de autores separados por `; ` |
| `ano` | Ano de publicação (4 dígitos, quando disponível) |
| `revista` | Nome do periódico, quando aplicável |
| `instituicao` | Instituição vinculada ao registro |
| `topicos` | Tópicos atribuídos ao registro pelo OpenAlex |
| `resumo` | Resumo / abstract |
| `doi` | DOI sem o prefixo `https://doi.org/` |
| `link_editor` | URL completa do DOI (link para o editor) |
| `acesso_aberto` | `True` se o registro está marcado como acesso aberto |
| `producao_nacional` | `True` se classificado como produção nacional |
| `revisado_por_pares` | `True` se classificado como revisado por pares |
| `termo_busca` | Termo da busca que originou a linha (útil em chamadas com lista de termos) |

## Uso Avançado

In [3]:
# Múltiplos termos de busca de uma vez
termos = ["judicialização da saúde", "natjus"]
dados_multi = scraper.raspar(pesquisa=termos, paginas=range(1, 2))

print(f"Total de registros (2 termos): {len(dados_multi)}")
dados_multi.groupby("termo_busca").size()

2026-05-18 19:07:08,061 - capes - INFO - Iniciando raspagem com parâmetros {'pesquisa': ['judicialização da saúde', 'natjus'], 'paginas': range(1, 2)}


2026-05-18 19:07:08,061 - capes - INFO - Iniciando raspagem para pesquisa=judicialização da saúde


2026-05-18 19:07:08,062 - capes - DEBUG - Definindo consulta


2026-05-18 19:07:08,062 - capes - DEBUG - {'q': 'all:contains(judicialização da saúde)', 'mode': 'advanced', 'source': 'all'}


2026-05-18 19:07:08,062 - capes - DEBUG - Definindo n_pags


...

Processando documentos:   0%|          | 0/1 [00:00<?, ?it/s]

Processando documentos: 100%|██████████| 1/1 [00:00<00:00, 16.21it/s]


2026-05-18 19:07:21,967 - capes - DEBUG - Adicionada coluna termo_busca=natjus aos resultados


Total de registros (2 termos): 50


termo_busca
judicialização da saúde    30
natjus                     20
dtype: int64

In [4]:
# Filtrar pelos rótulos do portal (acesso aberto + revisado por pares)
abertos = dados[dados["acesso_aberto"] & dados["revisado_por_pares"]]
print(f"Acesso aberto e revisado por pares: {len(abertos)} de {len(dados)}")
abertos[["titulo", "revista", "ano"]].head()

Acesso aberto e revisado por pares: 8 de 20


,titulo,revista,ano
0,Judicialização de produtos à base de canabidio...,Cadernos de Saúde Pública,2023
1,O direito à saúde e a dispensação judicial de ...,,2020
2,Notas técnicas para judicialização de anticoag...,Medicina (Ribeirão Preto),2022
6,Avaliação do risco de viés dos ensaios clínico...,Medicina (Ribeirão Preto),2023
13,PP47 Analysis Of Technical-Scientific Reports ...,International Journal of Technology Assessment...,2025


In [5]:
# Distribuição por ano
dados["ano"].value_counts().sort_index(ascending=False).head(10)

ano
2025    6
2024    3
2023    3
2022    3
2021    4
2020    1
Name: count, dtype: int64

## Exportação dos Dados

In [6]:
# Exportar para Excel
# dados.to_excel("capes_resultados.xlsx", index=False)

# Exportar para CSV
# dados.to_csv("capes_resultados.csv", index=False)